# Práctica 2 — Metaheurísticas  
# Práctica 2: Algoritmo Genético para Optimización de Hiperparámetros

## Optimización de hiperparámetros mediante Algoritmo Genético  
### Dataset: Wine Quality (Red Wine)

En esta práctica implementamos un **algoritmo genético** para optimizar los hiperparámetros de un modelo **RandomForestClassifier**.

El objetivo es **maximizar la accuracy** utilizando:

- Algoritmo Genético (AG)
- Random Search
- Grid Search

El fitness se calcula mediante **validación cruzada de 5 folds**, usando el accuracy medio.

A continuación se presenta el código completo, organizado y documentado.

# Código: Importaciones y carga del dataset

In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split

# cargar dataset
data = pd.read_csv("winequality-red.csv", sep=";")

# convertir problema a clasificación binaria
data["quality"] = (data["quality"] >= 6).astype(int)

X = data.drop("quality", axis=1)
y = data["quality"]

## Clase Param  
Esta clase representa un individuo del algoritmo genético.  
Es idéntica al archivo `params.py` original.

In [2]:
class Param:
    def __init__(
        self,
        n_estimators: int = 100,
        max_depth: int = 10,
        min_samples_split: int = 2,
        min_samples_leaf: int = 1,
        max_features: float = 1.0,
        bootstrap: int = 1,
        criterion: int = 0,
        class_weight: int = 0,
        max_leaf_nodes: int = 50,
        min_impurity_decrease: float = 0.0
    ):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.bootstrap = bootstrap
        self.criterion = criterion
        self.class_weight = class_weight
        self.max_leaf_nodes = max_leaf_nodes
        self.min_impurity_decrease = min_impurity_decrease
        self.validate_params()

    def validate_params(self):
        if not isinstance(self.n_estimators, int) or not (10 <= self.n_estimators <= 300):
            raise ValueError(f"n_estimators debe ser un entero entre 10 y 300. Recibido: {self.n_estimators}")

        if not isinstance(self.max_depth, int) or not (2 <= self.max_depth <= 30):
            raise ValueError(f"max_depth debe ser un entero entre 2 y 30. Recibido: {self.max_depth}")

        if not isinstance(self.min_samples_split, int) or not (2 <= self.min_samples_split <= 20):
            raise ValueError(f"min_samples_split debe ser un entero entre 2 y 20. Recibido: {self.min_samples_split}")

        if not isinstance(self.min_samples_leaf, int) or not (1 <= self.min_samples_leaf <= 20):
            raise ValueError(f"min_samples_leaf debe ser un entero entre 1 y 20. Recibido: {self.min_samples_leaf}")

        if not isinstance(self.max_features, (int, float)) or not (0.1 <= self.max_features <= 1.0):
            raise ValueError(f"max_features debe ser un real entre 0.1 y 1.0. Recibido: {self.max_features}")

        if self.bootstrap not in [0, 1]:
            raise ValueError(f"bootstrap debe ser binario (0 o 1). Recibido: {self.bootstrap}")

        if self.criterion not in [0, 1]:
            raise ValueError(f"criterion debe ser 0 (gini) o 1 (entropy). Recibido: {self.criterion}")

        if self.class_weight not in [0, 1]:
            raise ValueError(f"class_weight debe ser 0 (None) o 1 (balanced). Recibido: {self.class_weight}")

        if not isinstance(self.max_leaf_nodes, int) or not (10 <= self.max_leaf_nodes <= 200):
            raise ValueError(f"max_leaf_nodes debe ser un entero entre 10 y 200. Recibido: {self.max_leaf_nodes}")

        if not isinstance(self.min_impurity_decrease, (int, float)) or not (0.0 <= self.min_impurity_decrease <= 0.1):
            raise ValueError(f"min_impurity_decrease debe ser un real entre 0 y 0.1. Recibido: {self.min_impurity_decrease}")

    def to_list(self):
        return [
            self.n_estimators,
            self.max_depth,
            self.min_samples_split,
            self.min_samples_leaf,
            self.max_features,
            self.bootstrap,
            self.criterion,
            self.class_weight,
            self.max_leaf_nodes,
            self.min_impurity_decrease
        ]

## Función evaluate_solution_param  
Versión original del archivo `evaluate.py`, adaptada para Jupyter.  
Evalúa un objeto Param mediante validación cruzada.

In [3]:
def evaluate_solution_param(params):

    model = RandomForestClassifier(
        n_estimators=int(params.n_estimators),
        max_depth=int(params.max_depth),
        min_samples_split=int(params.min_samples_split),
        min_samples_leaf=int(params.min_samples_leaf),
        max_features=float(params.max_features),
        bootstrap=bool(params.bootstrap),
        criterion="gini" if params.criterion == 0 else "entropy",
        class_weight=None if params.class_weight == 0 else "balanced",
        max_leaf_nodes=int(params.max_leaf_nodes),
        min_impurity_decrease=float(params.min_impurity_decrease),
        random_state=42
    )

    scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
    return scores.mean()

## Generación de individuos  
Código idéntico al archivo `generateIndividual.py`.

In [4]:
import random

def generateIndividual():
    return Param(
        n_estimators=random.randint(10, 300),
        max_depth=random.randint(2, 30),
        min_samples_split=random.randint(2, 20),
        min_samples_leaf=random.randint(1, 20),
        max_features=random.uniform(0.1, 1.0),
        bootstrap=random.randint(0, 1),
        criterion=random.randint(0, 1),
        class_weight=random.randint(0, 1),
        max_leaf_nodes=random.randint(10, 200),
        min_impurity_decrease=random.uniform(0.0, 0.1)
    )

def bestIndividual():
    best = None
    best_fitness = -1.0

    for _ in range(5):
        actual_individual = generateIndividual()
        actual_fitness = evaluate_solution_param(actual_individual)

        if actual_fitness > best_fitness:
            best_fitness = actual_fitness
            best = actual_individual

    print(f"Best fitness of the five: {best_fitness}")
    return best

## Función metrics  
Código idéntico al archivo `metrics.py`.

In [5]:
import numpy as np

def metrics(fitness_scores):
    mean = np.mean(fitness_scores)
    standard_deviation = np.std(fitness_scores)
    best_absolute = np.max(fitness_scores)
    worst = np.min(fitness_scores)

    print("STATISTICAL RESULTS")
    print("="*30)
    print(f"Mean Accuracy: {mean:0.4f}")
    print(f"Standard Deviation: {standard_deviation:0.4f}")
    print(f"Best Overall Accuracy: {best_absolute:0.4f}")
    print(f"Worst Accuracy: {worst:0.4f}")

    return mean, standard_deviation, best_absolute, worst

## Grid Search  
Versión original del archivo `gridSearch.py`,  
pero adaptada para usar objetos Param en lugar de listas.

In [6]:
import itertools

def gridSearch(paramGrid):
    combinations = list(itertools.product(*paramGrid))
    
    totalComb = len(combinations)
    print(f"Total combinations to evaluate: {totalComb}")
    
    bestFitness = -1.0
    bestIndividual = None
    allFitnessScores = []
    
    for index, combo in enumerate(combinations):
        
        actualIndividual = Param(
            n_estimators=combo[0],
            max_depth=combo[1],
            min_samples_split=combo[2],
            min_samples_leaf=combo[3],
            max_features=combo[4],
            bootstrap=combo[5],
            criterion=combo[6],
            class_weight=combo[7],
            max_leaf_nodes=combo[8],
            min_impurity_decrease=combo[9]
        )
        
        actualFitness = evaluate_solution_param(actualIndividual)
        allFitnessScores.append(actualFitness)
        
        if actualFitness > bestFitness:
            bestFitness = actualFitness
            bestIndividual = actualIndividual
            
        if (index + 1) % 10 == 0 or (index + 1) == totalComb:
            print(f"[{index + 1}/{totalComb}] Evaluated -> Current Best: {bestFitness:.4f}")
            
    print("\n--- Grid Search Results ---")
    print(f"Best Accuracy: {bestFitness:.4f}")
    print(f"Best Parameters: {bestIndividual.to_list()}")
    
    print("\nOverall Statistics:")
    metrics(allFitnessScores)
    
    return bestFitness, bestIndividual

## Random Search  
Código idéntico al archivo `rs.py`, adaptado para usar evaluate_solution_param.

In [7]:
def random_search(iter):
    print(f" --- Random Search ( {iter} iter) --- ")

    mejor_individual_global = None
    mejor_fitness_global = -1.0
    fitness = []

    for i in range(iter):
        individual = bestIndividual()

        fitness_actual = evaluate_solution_param(individual)
        fitness.append(fitness_actual)

        if fitness_actual > mejor_fitness_global:
            mejor_fitness_global = fitness_actual
            mejor_individual_global = individual

        print(f"Iterations {i+1} / {iter} completed -> Fitness: {fitness_actual:.4f}")

    print(f"\n Best fitness (Random Search) : {mejor_fitness_global:.4f}")
    metrics(fitness)

    return mejor_fitness_global, mejor_individual_global

## Algoritmo Genético  
Código idéntico al archivo `evolutive.py`,  
solo adaptado para usar evaluate_solution_param.

In [8]:
import copy
import random

def crossover(param1, param2):
    return Param(
        param2.n_estimators,
        param1.max_depth,
        param1.min_samples_split,
        param1.min_samples_leaf,
        param1.max_features,
        param1.bootstrap,
        param2.criterion,
        param2.class_weight,
        param2.max_leaf_nodes,
        param2.min_impurity_decrease
    )

def mutation(param):
    nrand = random.randint(1,10)

    if nrand == 1:
        param.n_estimators = random.randint(10, 300)
    elif nrand == 2:
        param.max_depth = random.randint(2, 30)
    elif nrand == 3:
        param.min_samples_split = random.randint(2, 20)
    elif nrand == 4:
        param.min_samples_leaf = random.randint(1, 20)
    elif nrand == 5:
        param.max_features = random.uniform(0.1,1.0)
    elif nrand == 6:
        param.bootstrap = random.randint(0,1)
    elif nrand == 7:
        param.criterion = random.randint(0,1)
    elif nrand == 8:
        param.class_weight = random.randint(0,1)
    elif nrand == 9:
        param.max_leaf_nodes = random.randint(10, 200)
    elif nrand == 10:
        param.min_impurity_decrease = random.uniform(0,0.1)

def aplyMutation(childs, beta):
    for i in range(len(childs)):
        p = random.random()
        if p <= beta:
            mutation(childs[i])

def mergeArrays(a1, a2):
    result = []
    aux = []
    population = a1 + a2

    for ind in population:
        if ind[0] not in aux:
            aux.append(ind[0])
            result.append(ind)

    return result

def fitness(population):
    result = []
    for i in range(len(population)):
        result.append((population[i], evaluate_solution_param(population[i])))
    return result

def finalPopulation(population, childs):
    bests = mergeArrays(population, childs)
    bests = sorted(bests, key=lambda childs: childs[1], reverse=True)
    return bests[:len(population)]

def powerTournament(population, alpha, beta):
    childs = []
    for _ in range(len(population)//2):
        vp = population.copy()

        a = vp[random.randint(0, len(vp)-1)]
        vp.remove(a)
        b = vp[random.randint(0, len(vp)-1)]
        vp.remove(b)
        c = vp[random.randint(0, len(vp)-1)]

        a_fit = a[1]
        b_fit = b[1]
        c_fit = c[1]

        if a_fit > b_fit:
            if a_fit > c_fit:
                p1 = a[0]
            else:
                p1 = c[0]
        else:
            if b_fit > c_fit:
                p1 = b[0]
            else:
                p1 = c[0]

        vp = population.copy()

        a = vp[random.randint(0, len(vp)-1)]
        vp.remove(a)
        b = vp[random.randint(0, len(vp)-1)]
        vp.remove(b)
        c = vp[random.randint(0, len(vp)-1)]

        a_fit = a[1]
        b_fit = b[1]
        c_fit = c[1]

        if a_fit > b_fit:
            if a_fit > c_fit:
                p2 = a[0]
            else:
                p2 = c[0]
        else:
            if b_fit > c_fit:
                p2 = b[0]
            else:
                p2 = c[0]

        p = random.random()
        if p <= alpha:
            childs.append(crossover(p1, p2))
            childs.append(crossover(p2, p1))
        else:
            childs.append(copy.deepcopy(p1))
            childs.append(copy.deepcopy(p2))

    aplyMutation(childs, beta)
    childs = fitness(childs)
    return childs

def evolutive(Gen=20, alpha=0.8, beta=0.2):
    population = []

    for _ in range(20):
        individual = generateIndividual()
        population.append((individual, evaluate_solution_param(individual)))

    while Gen > 0:
        print(f"Gen Actual: {Gen}")
        Gen -= 1

        childs = powerTournament(population, alpha, beta)
        bests = finalPopulation(population, childs)
        population = bests

        print(f"Best Fitness : {population[0][1]}")

    return population

# **Comparación y análisis**

Para garantizar una evaluación rigurosa y equitativa entre los métodos estudiados, se ha igualado el **presupuesto computacional** del Algoritmo Genético y de la Búsqueda Aleatoria. El AG ejecuta **20 generaciones** con una población de **20 individuos**, lo que supone un total de:

$$
20 \text{ generaciones} \times 20 \text{ individuos} = 400 \text{ evaluaciones}
$$

A estas se añaden las evaluaciones iniciales de la población, alcanzando un total aproximado de **420 evaluaciones de fitness**.

De forma equivalente, la Búsqueda Aleatoria se ha configurado con **84 iteraciones**, ya que cada iteración evalúa **5 individuos** mediante la función `bestIndividual()`, lo que también suma:

$$
84 \text{ iteraciones} \times 5 \text{ evaluaciones} = 420 \text{ evaluaciones}
$$

Este ajuste permite comparar ambos métodos bajo un **mismo coste computacional**, eliminando sesgos derivados del número de evaluaciones.

Por otro lado, se ha incluido un **Grid Search** compuesto por **16 combinaciones** como baseline sistemático. La disparidad en el número de evaluaciones respecto al AG y al Random Search es **intencional**: aumentar la rejilla para igualar las 420 evaluaciones no solo sería computacionalmente prohibitivo, sino que **no garantizaría mejores resultados** debido a la rigidez inherente a un espacio de búsqueda fijo y discretizado.

Esta diferencia pone de manifiesto la **ineficiencia de los métodos exhaustivos** ante la explosión combinatoria generada por los **10 hiperparámetros** del Random Forest. En este contexto, la computación evolutiva demuestra ser una estrategia idónea, ya que permite explorar regiones amplias y continuas del espacio de búsqueda sin necesidad de evaluar todas las combinaciones posibles.

## Ejecución de Grid Search

In [9]:
myGrid = [
    [50, 150],
    [10, 20],
    [2],
    [1],
    [0.5, 1.0],
    [1],
    [0, 1],
    [0],
    [50],
    [0.0]
]

gridSearch(myGrid)

Total combinations to evaluate: 16
[10/16] Evaluated -> Current Best: 0.7342
[16/16] Evaluated -> Current Best: 0.7342

--- Grid Search Results ---
Best Accuracy: 0.7342
Best Parameters: [150, 10, 2, 1, 0.5, 1, 0, 0, 50, 0.0]

Overall Statistics:
STATISTICAL RESULTS
Mean Accuracy: 0.7264
Standard Deviation: 0.0031
Best Overall Accuracy: 0.7342
Worst Accuracy: 0.7217


(np.float64(0.7342221786833856), <__main__.Param at 0x19044961b50>)

# Función Random Search
Nota: La función random_search ya llama internamente a metrics(),  por lo que imprime en pantalla el "Mean Accuracy" y la "Standard Deviation"

In [10]:
# --- Celda para ejecutar Random Search y obtener datos para la memoria ---
print("INICIANDO COMPARATIVA: RANDOM SEARCH")
# Ejecutamos 100 iteraciones para una exploración significativa del espacio
mejor_fitness_rs, mejor_individual_rs = random_search(100)

INICIANDO COMPARATIVA: RANDOM SEARCH
 --- Random Search ( 100 iter) --- 
Best fitness of the five: 0.7229682601880877
Iterations 1 / 100 completed -> Fitness: 0.7230
Best fitness of the five: 0.7317084639498433
Iterations 2 / 100 completed -> Fitness: 0.7317
Best fitness of the five: 0.716087382445141
Iterations 3 / 100 completed -> Fitness: 0.7161
Best fitness of the five: 0.6917045454545455
Iterations 4 / 100 completed -> Fitness: 0.6917
Best fitness of the five: 0.7348295454545454
Iterations 5 / 100 completed -> Fitness: 0.7348
Best fitness of the five: 0.7079663009404389
Iterations 6 / 100 completed -> Fitness: 0.7080
Best fitness of the five: 0.6879369122257053
Iterations 7 / 100 completed -> Fitness: 0.6879
Best fitness of the five: 0.7304565047021943
Iterations 8 / 100 completed -> Fitness: 0.7305
Best fitness of the five: 0.7379584639498432
Iterations 9 / 100 completed -> Fitness: 0.7380
Best fitness of the five: 0.703569749216301
Iterations 10 / 100 completed -> Fitness: 0.703

In [11]:
# Función Random Search con 20 iteraciones
mejor_fitness_rs, mejor_individual_rs = random_search(20)

 --- Random Search ( 20 iter) --- 
Best fitness of the five: 0.7142045454545455
Iterations 1 / 20 completed -> Fitness: 0.7142
Best fitness of the five: 0.6891849529780564
Iterations 2 / 20 completed -> Fitness: 0.6892
Best fitness of the five: 0.7298256269592476
Iterations 3 / 20 completed -> Fitness: 0.7298
Best fitness of the five: 0.7329643416927899
Iterations 4 / 20 completed -> Fitness: 0.7330
Best fitness of the five: 0.7248354231974922
Iterations 5 / 20 completed -> Fitness: 0.7248
Best fitness of the five: 0.7342163009404389
Iterations 6 / 20 completed -> Fitness: 0.7342
Best fitness of the five: 0.717962382445141
Iterations 7 / 20 completed -> Fitness: 0.7180
Best fitness of the five: 0.7142025862068965
Iterations 8 / 20 completed -> Fitness: 0.7142
Best fitness of the five: 0.726712382445141
Iterations 9 / 20 completed -> Fitness: 0.7267
Best fitness of the five: 0.7235775862068965
Iterations 10 / 20 completed -> Fitness: 0.7236
Best fitness of the five: 0.7267104231974921
I

### Comparativa de Esfuerzo Computacional Equivalente: Random Search (Hacemos 84 iteraciones)

Para garantizar una **comparativa rigurosa y justa** entre el Algoritmo Genético (AG) y la Búsqueda Aleatoria (RS), se ha igualado el número total de evaluaciones de la función de fitness (presupuesto computacional):

1. **Algoritmo Genético:** Con una población de 20 individuos y 20 generaciones, el esfuerzo total es de aproximadamente **420 evaluaciones** (20 iniciales + 20x20 evolutivas) [2, 3].
2. **Random Search:** Dado que cada iteración de la función `random_search` evalúa a **5 individuos** mediante la función `bestIndividual()` [4], realizaremos **84 iteraciones** ($84 \times 5 = 420$).

De este modo, cualquier diferencia en el accuracy obtenido se deberá exclusivamente a la **eficiencia de la estrategia de búsqueda** (evolutiva vs. ciega) y no a una diferencia en el tiempo de cómputo permitido.


In [12]:
# Función Random Search con 20 iteraciones
mejor_fitness_rs, mejor_individual_rs = random_search(84)

 --- Random Search ( 84 iter) --- 
Best fitness of the five: 0.7367241379310345
Iterations 1 / 84 completed -> Fitness: 0.7367
Best fitness of the five: 0.6898079937304076
Iterations 2 / 84 completed -> Fitness: 0.6898
Best fitness of the five: 0.6935736677115988
Iterations 3 / 84 completed -> Fitness: 0.6936
Best fitness of the five: 0.732962382445141
Iterations 4 / 84 completed -> Fitness: 0.7330
Best fitness of the five: 0.7123354231974922
Iterations 5 / 84 completed -> Fitness: 0.7123
Best fitness of the five: 0.7035717084639499
Iterations 6 / 84 completed -> Fitness: 0.7036
Best fitness of the five: 0.7279741379310345
Iterations 7 / 84 completed -> Fitness: 0.7280
Best fitness of the five: 0.7423452194357367
Iterations 8 / 84 completed -> Fitness: 0.7423
Best fitness of the five: 0.7260913009404388
Iterations 9 / 84 completed -> Fitness: 0.7261
Best fitness of the five: 0.6935717084639498
Iterations 10 / 84 completed -> Fitness: 0.6936
Best fitness of the five: 0.7279565047021944


## Función evolutiveTest  
Código equivalente al archivo `geneticAlgorithm.py`,  
adaptado mínimamente para funcionar en Jupyter.

In [13]:
import numpy as np

def evolutiveTest():

    population = evolutive(beta=0.5, alpha=0.6)

    fitnesses = []
    for i in range(len(population)):
        fitnesses.append(population[i][1])

    print("\n--- Results ---")
    print(f"Best Individual Score: {fitnesses[0]}")
    print(f"Population Mean: {np.mean(fitnesses)}")
    print(f"Population Desviation: {np.std(fitnesses)}")

## Ejecución del Algoritmo Genético  
En Jupyter no se usa `if __name__ == "__main__":`,  
así que simplemente llamamos a la función.

In [14]:
evolutiveTest()

Gen Actual: 20
Best Fitness : 0.7310932601880877
Gen Actual: 19
Best Fitness : 0.7335952194357367
Gen Actual: 18
Best Fitness : 0.7335952194357367
Gen Actual: 17
Best Fitness : 0.73608934169279
Gen Actual: 16
Best Fitness : 0.7367163009404389
Gen Actual: 15
Best Fitness : 0.7423510971786834
Gen Actual: 14
Best Fitness : 0.7423510971786834
Gen Actual: 13
Best Fitness : 0.7423510971786834
Gen Actual: 12
Best Fitness : 0.7423510971786834
Gen Actual: 11
Best Fitness : 0.7423510971786834
Gen Actual: 10
Best Fitness : 0.7429741379310345
Gen Actual: 9
Best Fitness : 0.7429741379310345
Gen Actual: 8
Best Fitness : 0.7429741379310345
Gen Actual: 7
Best Fitness : 0.7442260971786834
Gen Actual: 6
Best Fitness : 0.7442260971786834
Gen Actual: 5
Best Fitness : 0.7442260971786834
Gen Actual: 4
Best Fitness : 0.7442260971786834
Gen Actual: 3
Best Fitness : 0.7442260971786834
Gen Actual: 2
Best Fitness : 0.7442260971786834
Gen Actual: 1
Best Fitness : 0.7442260971786834

--- Results ---
Best Individua